In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
import sys

sys.path.append("../02-encoding")
sys.path.append('./retrieval/')

In [4]:
from pkgimp import *
from bson import ObjectId
from tqdm import tqdm
import torch

from nb2p import database, fileop, config, astparse
from nb2p.notebook import Notebook

In [5]:
DATASET_NAME = 'distilkaggle'
# DATASET_NAME = 'pmbf'
# DATASET_NAME = 'dspipeline2'

In [6]:
ground_truth = fileop.read_json(os.path.join(DATASET_NAME, f"{DATASET_NAME}_groundtruth.json"))
ground_truth[17]

{'func_defs': ["def most_popular(group, n_max=5):\n    relevance = group['relevance'].values\n    hotel_cluster = group['hotel_cluster'].values\n    most_popular = hotel_cluster[np.argsort(relevance)[::-1]][:n_max]\n    return np.array_str(most_popular)[1:-1]"],
 'code_lines': ['print(check_output(["ls", "../input"]).decode("utf8"))',
  "train = pd.read_csv('../input/train.csv',",
  "                    dtype={'is_booking':bool,'srch_destination_id':np.int32, 'hotel_cluster':np.int32},",
  "                    usecols=['srch_destination_id','is_booking','hotel_cluster'],",
  '                    chunksize=1000000)',
  'aggs = []',
  "print('-'*38)",
  'for chunk in train:',
  "    agg = chunk.groupby(['srch_destination_id',",
  "                         'hotel_cluster'])['is_booking'].agg(['sum','count'])",
  '    agg.reset_index(inplace=True)',
  '    aggs.append(agg)',
  "    print('.',end='')",
  "print('')",
  'aggs = pd.concat(aggs, axis=0)',
  'aggs.head()',
  'CLICK_WEIGHT = 0.0

In [7]:
from tokenizers import Tokenizer
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig, RobertaForSequenceClassification
from nb2p.dfgtree.compressor.model import Model

def get_llm_prompt(encoding: dict):
    result = ""
    
    if len(encoding['func_defs']) == 0:
        result += 'The notebook has no global function definitions.\n'
    else:
        result += 'The notebook has the following global function definitions:\n'
        for fdef in encoding['func_defs']:
            result += '### FUNCTION DEFINITION\n'
            result += fdef
            result += '\n'
            result += '### FUNCTION DEFINITION END HERE\n'

    result += 'And the notebook code is:\n'
    result += '### NOTEBOOK CODE\n'
    result += "\n".join(encoding['code_lines'])
    result += '\n'
    result += '### NOTEBOOK CODE END HERE\n'

    result += 'Hence, the pipeline components are:\n'
    result += '### COMPONENT'

    return result

print(get_llm_prompt(ground_truth[17]))

/home/haotian/r/ascent/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The notebook has the following global function definitions:
### FUNCTION DEFINITION
def most_popular(group, n_max=5):
    relevance = group['relevance'].values
    hotel_cluster = group['hotel_cluster'].values
    most_popular = hotel_cluster[np.argsort(relevance)[::-1]][:n_max]
    return np.array_str(most_popular)[1:-1]
### FUNCTION DEFINITION END HERE
And the notebook code is:
### NOTEBOOK CODE
print(check_output(["ls", "../input"]).decode("utf8"))
train = pd.read_csv('../input/train.csv',
                    dtype={'is_booking':bool,'srch_destination_id':np.int32, 'hotel_cluster':np.int32},
                    usecols=['srch_destination_id','is_booking','hotel_cluster'],
                    chunksize=1000000)
aggs = []
print('-'*38)
for chunk in train:
    agg = chunk.groupby(['srch_destination_id',
                         'hotel_cluster'])['is_booking'].agg(['sum','count'])
    agg.reset_index(inplace=True)
    aggs.append(agg)
    print('.',end='')
print('')
aggs = pd.concat(agg

### Retrieve Few-shot Prompting Samples

Since we need to retrieve by finding the NN from the GraphCodeBERT embeddings, we need to first initialize the model.

In [8]:
def get_xs_model(model_dir: str, size: int):
    config = RobertaConfig.from_pretrained("microsoft/graphcodebert-base")
    config.num_attention_heads = 8
    config.hidden_size = 96
    config.intermediate_size = 64
    config.vocab_size = 1000
    config.num_hidden_layers = 12
    config.hidden_dropout_prob = 0.2

    tokenizer_path = os.path.join("../02-encoding/compressor", "BPE" + "_" + str(config.vocab_size) + ".json")
    tokenizer = Tokenizer.from_file(tokenizer_path)

    model = Model(RobertaForSequenceClassification(config=config), config, tokenizer)

    model_dir = os.path.join(model_dir, str(size), "model.bin")
    model.load_state_dict(torch.load(model_dir))

    return model, tokenizer

In [9]:
DEVICE = torch.device("cuda")

model, tokenizer = get_xs_model('../02-encoding/compressor/GraphCodeBERT/clone_detection/checkpoint', 3)
model.eval()
model = model.to(DEVICE)

Next, set up parser to initialize the encoding builder.

In [10]:
parser, lang = astparse.parser()

In [11]:
from nb2p.dfgtree.dfg import CodeEncodingBuilder

encoding_builder = CodeEncodingBuilder(tokenizer, model)
encoding_builder

<CodeEncodingBuilder device=cuda>

Initialize ChromaDB embedding function.

In [24]:
from chromadb import Documents, EmbeddingFunction, Embeddings
import logging
logging.getLogger().setLevel(logging.WARN)


class GCBEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        result = []
        for t in input:
            inp = encoding_builder.make_input(t)
            out = encoding_builder.get_encoding(inp).detach().cpu()[0].numpy()
            result.append(out)

        return result

Query and store few-shot data

In [25]:
import chromadb
import time
chroma_client = chromadb.HttpClient(port=8001)

REF_DATASET_NAME = 'distilkaggle'

collection = chroma_client.get_or_create_collection(name=REF_DATASET_NAME, embedding_function=GCBEmbeddingFunction())
collection

ground_truth = fileop.read_json(os.path.join(DATASET_NAME, f"{DATASET_NAME}_groundtruth.json"))
ground_truth[17]
test_code = "\n".join(ground_truth[1]['code_lines'])

few_shot_data = []
for i in range(len(ground_truth)):
    test_code = "\n".join(ground_truth[i]['code_lines'])
    try:
        res = collection.query(
            query_texts=[test_code],
            n_results=3
        )
    except Exception as e: 
        # rare exception, e.g. pmbf groundtruth[295, 316,etc]
        # https://github.com/chroma-core/chroma/issues/868
        res = collection.peek(3)
        res['embeddings']=None
        print(f"Exception in collection.query {i}:\n {e}")
        pass
    few_shot_data.append(res)

fileop.write_json(few_shot_data, os.path.join(f"{DATASET_NAME}_fewshot_data.json"))

## Build few-shot prompt

In [35]:
# LLM_PROMPT_PREFIX = """Suppose you have a data science notebook and want to extract pipeline components based on their semantic purposes. There are two requirements. First, each component should contain consecutive code, one more more lines, in the notebook. You should output one code cell for each component. You cannot modify, swap, or exclude any code. Second, each component should represent a specific stage in the data science process. Components can have the same stage. The example stages are:
# - data acquisition (such as load, collect, obtain, capture, survey)
# - data preparation (such as explore, wrangle, clean, filter, organize)
# - storage (such as preserve, archive, warehouse, log, recycle)
# - feature engineering (such as feature, label, annotate)
# - modeling (such as classify, cluster, mine, analyze, process)
# - training (such as tune, optimize)
# - evaluation (such as validate, test, verify, review)
# - prediction (such as discover, derive, determine)
# - interpretation (such as transform, visualize, render, translate, explain)
# - communication (such as transfer, share, distribute, transmit, publish)

# """

LLM_PROMPT_PREFIX = """Suppose you have a data science notebook and want to extract code for pipeline components based on their semantic purposes. Just output one code snippet, beginning with '```python\\n' and ends with '```' for each component's original code. Do not summarize its purpose, just output its code. There are two requirements. First, each component should contain consecutive code, one or more lines, as those in the given notebook. You cannot modify, swap, or exclude any code in the given notebook. Second, each component should represent a specific stage in the data science process. Components can have the same stage. The example stages are:
- data acquisition (such as load, collect, obtain, capture, survey)
- data preparation (such as explore, wrangle, clean, filter, organize)
- storage (such as preserve, archive, warehouse, log, recycle)
- feature engineering (such as feature, label, annotate)
- modeling (such as classify, cluster, mine, analyze, process)
- training (such as tune, optimize)
- evaluation (such as validate, test, verify, review)
- prediction (such as discover, derive, determine)
- interpretation (such as transform, visualize, render, translate, explain)
- communication (such as transfer, share, distribute, transmit, publish)

"""

DEMONSTRATION_PREFIX= '\nThe following shows the extracted pipeline components from some demonstrative notebooks\n'

In [36]:
import ast

def get_segment_code(code: str, segment_ends: List[int], parser: Parser):
    code_bytes = bytes(code, "utf8")
    parse_tree = parser.parse(code_bytes, encoding='utf8')
    cursor = parse_tree.walk()

    result = []
    curr_code = ""
    curr_index = 0

    cursor.goto_first_child()
    curr_ast_children = 0

    if segment_ends[curr_index] == curr_ast_children:
        yield code[cursor.node.start_byte:cursor.node.end_byte]
        curr_index += 1
    
    while cursor.goto_next_sibling():
        curr_ast_children += 1
        # distilkaggle 159, 269, ...
        if curr_index < len(segment_ends) and segment_ends[curr_index] == curr_ast_children:
            curr_code += code_bytes[cursor.node.start_byte:cursor.node.end_byte].decode()
            yield curr_code
            
            curr_code = ""
            curr_index += 1
        else:
            curr_code += code_bytes[cursor.node.start_byte:cursor.node.end_byte].decode()
            curr_code += '\n'

    return

test_segment_code = list(get_segment_code(few_shot_data[1]['documents'][0][0], ast.literal_eval(few_shot_data[1]['metadatas'][0][0].get('segment_ends')), parser))
for c in test_segment_code:
    print(c)
    print("---")

print(os.listdir("../input"))
dataset = pd.read_csv('../input/Mall_Customers.csv',index_col='CustomerID')
---
dataset.head()
dataset.info()
dataset.describe()
---
dataset.groupby(by='Genre')['Age'].count()
---
dataset.isnull().sum()
dataset.drop_duplicates(inplace=True)
X = dataset.iloc[:, [2, 3]].values
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters = i, init = 'k-means++', random_state = 42)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)
---
plt.figure(figsize=(10,5))
sns.lineplot(range(1, 11), wcss,marker='o',color='red')
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()
kmeans = KMeans(n_clusters = 5, init = 'k-means++', random_state = 42)
y_kmeans = kmeans.fit_predict(X)
---


In [37]:
def reconstruct_annotation(raw_code: list, segment_ends: list):
    if isinstance(raw_code, str):
        code_lines = raw_code.split("\n")
    else:
        code_lines = raw_code

    if isinstance(segment_ends, str):
        segment_ends = ast.literal_eval(segment_ends)

    annotated_code = []
    # prev_end = 0
    # for idx, segment_end in enumerate(segment_ends):
    #     if int(segment_end) == 0: 
    #         continue # some segment end = 0 in the data ?
    #     segment = "\n".join(code_lines[prev_end:int(segment_end) + 1])
    #     annotated_code.append(f"### Component:\n```python\n{segment}\n```")
    #     prev_end = int(segment_end) + 1

    for segment in get_segment_code("\n".join(code_lines), segment_ends, parser):
        annotated_code.append(f"### Component:\n```python\n{segment}\n```")

    llm_output = "\n\n".join(annotated_code)
    return llm_output

In [38]:
few_shot_data = fileop.read_json(os.path.join(f"{DATASET_NAME}_fewshot_data.json"))

In [39]:
def build_few_short_sample(few_shot_sample, num_samples=2):
    num_samples = min(num_samples, len(few_shot_sample['documents'][0]))
    # print (num_samples)
    few_shot_prompt = DEMONSTRATION_PREFIX
    for i in range(num_samples):
        if len(few_shot_sample['documents']) == 1:
            raw_ground_truth = few_shot_sample['documents'][0][i]
        else:
            raw_ground_truth = few_shot_sample['documents'][i]
        if len(few_shot_sample['metadatas']) == 1:
            ref_segment_ends = few_shot_sample['metadatas'][0][i].get('segment_ends')
        else:
            ref_segment_ends = few_shot_sample['metadatas'][i].get('segment_ends')
        few_shot_prompt += "### Extract pipeline components based on notebook's semantic purposes\n"
        few_shot_prompt += '### SAMPLE NOTEBOOK CODE\n'
        few_shot_prompt += raw_ground_truth
        few_shot_prompt += '\n### SAMPLE NOTEBOOK CODE END HERE\n'
        few_shot_prompt += '### SAMPLE EXTRACTED COMPONENTS\n'
        few_shot_prompt += reconstruct_annotation(raw_ground_truth, ref_segment_ends)
        few_shot_prompt += '\n### SAMPLE EXTRACTED COMPONENTS END HERE\n'
        few_shot_prompt += '\nEND_OF_DEMO\n\n'
        
    return few_shot_prompt


def make_llm_prompt(encoding: dict, cot = False, few_shot_sample = None):
    result = LLM_PROMPT_PREFIX
    if few_shot_sample:
        result += build_few_short_sample(few_shot_sample)
        result += "### Extract pipeline components based on notebook's semantic purposes\n"
    if len(encoding['func_defs']) == 0:
        result += 'The notebook has no global function definitions.\n'
    else:
        result += 'The notebook has the following global function definitions:\n'
        for fdef in encoding['func_defs']:
            result += '### FUNCTION DEFINITION\n'
            result += fdef
            result += '\n'
            result += '### FUNCTION DEFINITION END HERE\n'

    result += 'And the notebook code is:\n'
    result += '### NOTEBOOK CODE\n'
    result += "\n".join(encoding['code_lines'])
    result += '\n'
    result += '### NOTEBOOK CODE END HERE\n'

    result += 'Hence, the pipeline components are:\n'
    result += '### COMPONENT'

    if cot:
        result += "\nLet's think step by step."
    return result

In [40]:
prompt = make_llm_prompt(ground_truth[1], few_shot_sample=few_shot_data[1])
print(prompt)

Suppose you have a data science notebook and want to extract code for pipeline components based on their semantic purposes. Just output one code snippet, beginning with '```python\n' and ends with '```' for each component's original code. Do not summarize its purpose, just output its code. There are two requirements. First, each component should contain consecutive code, one or more lines, as those in the given notebook. You cannot modify, swap, or exclude any code in the given notebook. Second, each component should represent a specific stage in the data science process. Components can have the same stage. The example stages are:
- data acquisition (such as load, collect, obtain, capture, survey)
- data preparation (such as explore, wrangle, clean, filter, organize)
- storage (such as preserve, archive, warehouse, log, recycle)
- feature engineering (such as feature, label, annotate)
- modeling (such as classify, cluster, mine, analyze, process)
- training (such as tune, optimize)
- e

## Build prompt and store

In [41]:
ground_truth = fileop.read_json(os.path.join(DATASET_NAME, f"{DATASET_NAME}_groundtruth.json"))
few_shot_data = few_shot_data = fileop.read_json(os.path.join(f"{DATASET_NAME}_fewshot_data.json"))
# print(ground_truth[1])
prompt = make_llm_prompt(ground_truth[1], few_shot_sample=few_shot_data[1])
# print(prompt)
few_shot_prompt = []
for i in range(len(ground_truth)):
    try:
        few_shot_prompt.append(make_llm_prompt(ground_truth[i], few_shot_sample=few_shot_data[i]))
    except Exception as e:
        print(f"failed to process sample {i}")
fileop.write_json(few_shot_prompt, os.path.join(f"{DATASET_NAME}_2shot_prompt.json"))

In [43]:
make_llm_prompt(ground_truth[295], few_shot_sample=few_shot_data[295])

'Suppose you have a data science notebook and want to extract pipeline components based on their semantic purposes. There are two requirements. First, each component should contain consecutive code, one more more lines, in the notebook. You should output one code cell for each component. You cannot modify, swap, or exclude any code. Second, each component should represent a specific stage in the data science process. Components can have the same stage. The example stages are:\n- data acquisition (such as load, collect, obtain, capture, survey)\n- data preparation (such as explore, wrangle, clean, filter, organize)\n- storage (such as preserve, archive, warehouse, log, recycle)\n- feature engineering (such as feature, label, annotate)\n- modeling (such as classify, cluster, mine, analyze, process)\n- training (such as tune, optimize)\n- evaluation (such as validate, test, verify, review)\n- prediction (such as discover, derive, determine)\n- interpretation (such as transform, visualize,